# Overlap-output comparison

Select a GPU runtime and run the first three code cells. The notebook trains one new **patch_overlap** model from scratch, with the original experiment's data and training settings. It reuses the four completed comparison results and their shared feature encoder.

Each time-expert token predicts 16 points instead of 8, with normalized weighted overlap-add. Input patching, frequency expert, and router remain the same. Results are saved in a separate **overlap_output** folder. The comparison reports generation metrics, patch-boundary changes, and within-window variation.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import getpass
import io
import json
import os
import subprocess
import sys
import urllib.error
import urllib.request
import zipfile

drive.mount('/content/drive')
REPOSITORY = 'Zhou-Leheng/adaptive-time-frequency-generation'
request = urllib.request.Request(
    f'https://api.github.com/repos/{REPOSITORY}/zipball/main',
    headers={'Accept': 'application/vnd.github+json'},
)
try:
    response = urllib.request.urlopen(request, timeout=90)
except urllib.error.HTTPError as error:
    if error.code not in (401, 404):
        raise
    try:
        github_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        github_token = getpass.getpass('GitHub token with repository contents read access: ')
    request.add_header('Authorization', f'Bearer {github_token}')
    response = urllib.request.urlopen(request, timeout=90)
    del github_token

with response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
del request, response
root = Path('/content/atfg_code')
root.mkdir(exist_ok=True)
for member in archive.infolist():
    target = (root / member.filename).resolve()
    if not target.is_relative_to(root.resolve()):
        raise ValueError('Unexpected archive path')
archive.extractall(root)
CODE = root / archive.namelist()[0].split('/')[0]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-benchmark.txt')])
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before starting the experiment.'
print('GPU:', torch.cuda.get_device_name(0))
print('Code:', CODE)

In [ ]:
RUN_NAME = 'etth1_seed42'
ABLATION_NAME = 'overlap_output'
BENCHMARK_NAME = 'benchmark_diffusion_ts'
RESULTS = Path('/content/drive/MyDrive/ATFG') / RUN_NAME
assert (RESULTS / 'settings.json').is_file(), 'Select the completed original fusion experiment.'
benchmark = RESULTS / BENCHMARK_NAME
assert (benchmark / 'comparison.json').is_file(), 'Finish the Diffusion-TS comparison first.'
original_rows = json.loads((benchmark / 'comparison.json').read_text())
assert {'fixed', 'global', 'patch', 'diffusion_ts'} <= {row['method'] for row in original_rows}, 'Wait for all four original methods to finish evaluation.'
settings = json.loads((RESULTS / 'settings.json').read_text())
OUTPUT = RESULTS / ABLATION_NAME
LOG = Path('/content') / f'atfg_overlap_{RUN_NAME}.log'
PID_RECORD = Path('/content') / f'atfg_overlap_{RUN_NAME}_pid.json'
print('Source experiment:', RESULTS)
print('Train one new model:', settings['steps'], 'steps; seed', settings['seed'])
print('Length:', settings['length'], 'LR:', settings['lr'], 'batch:', settings['batch_size'])
print('Comparison output:', OUTPUT)


In [ ]:
from filelock import FileLock
with FileLock(str(RESULTS / '.run.lock'), timeout=0):
    pass
command = [
    'nohup', sys.executable, '-u', str(CODE / 'run_overlap.py'),
    '--results', str(RESULTS), '--name', ABLATION_NAME,
    '--benchmark-name', BENCHMARK_NAME,
    '--download', '--data', '/content/ETTh1.csv',
]
with LOG.open('a') as handle:
    process = subprocess.Popen(command, cwd=CODE, stdout=handle, stderr=subprocess.STDOUT,
                               stdin=subprocess.DEVNULL, start_new_session=True)
PID_RECORD.write_text(json.dumps({'pid': process.pid, 'script': str(CODE / 'run_overlap.py')}))
print('Overlap PID:', process.pid)
print('Rerun the next cell to read the log.')


In [ ]:
if LOG.exists():
    print(LOG.read_text(errors='replace')[-8000:])
else:
    print('The log has not been created yet.')


In [ ]:
import pandas as pd
from IPython.display import display
summary = OUTPUT / 'comparison.csv'
if summary.exists():
    frame = pd.read_csv(summary)
    display(frame[['method', 'context_fid', 'discriminative_mean', 'predictive_mean',
                   'correlational_score', 'patch_boundary_ratio', 'mean_within_window_std']])
    if 'patch_overlap' not in set(frame['method']):
        print('Original results loaded. The new overlap evaluation is still pending.')
    print('Complete CSV:', summary)
else:
    print('The new model is still training or sampling. Read the log above.')


In [ ]:
from IPython.display import Image, display
for name in ['samples.png', 'router.png']:
    path = OUTPUT / 'patch_overlap' / name
    if path.exists():
        display(Image(filename=str(path)))


The boundary ratio compares changes at each original 8-point boundary with changes at other positions. Read it together with quality metrics and within-window variation. The same launch command resumes saved training checkpoints and skips completed evaluation stages.

To stop at a safe boundary, set STOP = True below and run the cell.


In [ ]:
STOP = False
if STOP:
    import signal
    record = json.loads(PID_RECORD.read_text())
    command_line = Path(f"/proc/{record['pid']}/cmdline")
    if command_line.exists() and record['script'].encode() in command_line.read_bytes().split(b'\x00'):
        os.kill(record['pid'], signal.SIGTERM)
        print('Stop requested. Read the log for the checkpoint confirmation.')
    else:
        print('The recorded process has already exited.')
